# 📘 연산자와 해시

**operator** 모듈은 연산자를 함수로, **hashlib**은 데이터 무결성과
보안을 위한 해시 함수를 제공합니다.

**학습 목표:**
- operator: 연산자 함수, itemgetter, attrgetter
- hashlib: SHA-256, MD5, HMAC
- 파일 해시와 비밀번호 해싱

## 1. operator 모듈

`operator`는 연산자를 함수 형태로 제공합니다.
함수형 프로그래밍과 `sorted()`, `reduce()` 등에서 유용합니다.

In [ ]:
import operator
from functools import reduce

# ┌─────────────────────────────────────────┐
# │  operator 모듈 주요 함수                  │
# │  operator.add(a, b)    → a + b           │
# │  operator.sub(a, b)    → a - b           │
# │  operator.mul(a, b)    → a * b            │
# │  operator.eq(a, b)     → a == b           │
# │  operator.itemgetter   → 딕셔너리/리스트 접근│
# │  operator.attrgetter   → 속성 접근          │
# └─────────────────────────────────────────┘

# 산술 연산자
print(f"add(3, 5): {operator.add(3, 5)}")
print(f"sub(10, 3): {operator.sub(10, 3)}")
print(f"mul(4, 5): {operator.mul(4, 5)}")
print(f"truediv(10, 3): {operator.truediv(10, 3):.2f}")

# 비교 연산자
print(f"\neq(3, 3): {operator.eq(3, 3)}")
print(f"lt(3, 5): {operator.lt(3, 5)}")
print(f"gt(5, 3): {operator.gt(5, 3)}")

# reduce와 함께
numbers = [1, 2, 3, 4, 5]
total = reduce(operator.add, numbers)
print(f"\nreduce(add, [1..5]): {total}")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  itemgetter와 attrgetter                  │
# │  itemgetter → 인덱스/키로 접근              │
# │  attrgetter → 속성명으로 접근               │
# │  sorted()의 key 함수로 유용                  │
# └─────────────────────────────────────────┘

students = [
    {"name": "김파이", "age": 25, "score": 92},
    {"name": "이코딩", "age": 23, "score": 88},
    {"name": "박개발", "age": 27, "score": 95},
]

# itemgetter로 정렬
by_score = sorted(students, key=operator.itemgetter("score"))
print("점수순 정렬:")
for s in by_score:
    print(f"  {s['name']}: {s['score']}점")

by_age = sorted(students, key=operator.itemgetter("age"))
print("\n나이순 정렬:")
for s in by_age:
    print(f"  {s['name']}: {s['age']}세")

# attrgetter 예시
class Product:
    def __init__(self, name, price):
        self.name = name
        self.price = price
    def __repr__(self):
        return f"{self.name}({self.price}원)"

products = [Product("노트북", 1500000), Product("폰", 800000), Product("태블릿", 500000)]
by_price = sorted(products, key=operator.attrgetter("price"))
print(f"\n가격순 정렬: {by_price}")

## 2. hashlib 모듈 — 해시 함수

`hashlib`은 데이터의 무결성 검증과 비밀번호 해싱에 사용합니다.

In [ ]:
import hashlib
import hmac

# ┌─────────────────────────────────────────┐
# │  hashlib 주요 함수                        │
# │  hashlib.sha256()  → SHA-256 해시          │
# │  hashlib.md5()     → MD5 해시              │
# │  .hexdigest()      → 16진수 문자열          │
# │  .update()         → 점진적 해시 갱신       │
# └─────────────────────────────────────────┘

# 사용 가능한 해시 알고리즘
print(f"사용 가능한 알고리즘: {', '.join(sorted(hashlib.algorithms_available))[:50]}...")

# SHA-256 해시
message = "Hello, Python!"
sha256 = hashlib.sha256(message.encode()).hexdigest()
print(f"\n메시지: {message}")
print(f"SHA-256: {sha256}")

# MD5 해시
md5 = hashlib.md5(message.encode()).hexdigest()
print(f"MD5: {md5}")

# 점진적 해시 (대용량 파일용)
chunk_hash = hashlib.sha256()
chunk_hash.update(b"Hello, ")
chunk_hash.update(b"Python!")
print(f"\n점진적 해시: {chunk_hash.hexdigest()}")

# HMAC (키드 해시)
key = b"secret_key"
hmac_hash = hmac.new(key, message.encode(), hashlib.sha256).hexdigest()
print(f"HMAC-SHA256: {hmac_hash[:32]}...")

In [ ]:
# ┌─────────────────────────────────────────┐
# │  실용 예제: 비밀번호 해싱과 파일 무결성     │
# │  비밀번호 → salt + SHA-256 (실제는 bcrypt) │
# │  파일     → SHA-256로 무결성 확인           │
# └─────────────────────────────────────────┘

import hashlib
import secrets
import tempfile
from pathlib import Path

# 비밀번호 해싱 (실제로는 bcrypt 권장)
def hash_password(password: str, salt: str = None) -> tuple:
    """비밀번호를 해싱합니다. (salt, hashed) 반환"""
    if salt is None:
        salt = secrets.token_hex(16)
    hashed = hashlib.sha256((password + salt).encode()).hexdigest()
    return salt, hashed

def verify_password(password: str, salt: str, hashed: str) -> bool:
    """비밀번호를 검증합니다."""
    _, new_hash = hash_password(password, salt)
    return new_hash == hashed

# 비밀번호 해싱
salt, hashed = hash_password("my_password123")
print(f"Salt: {salt[:16]}...")
print(f"Hashed: {hashed[:32]}...")

# 검증
print(f"\n올바른 비밀번호: {verify_password('my_password123', salt, hashed)}")
print(f"잘못된 비밀번호: {verify_password('wrong_password', salt, hashed)}")

# 파일 해시
tmp = Path(tempfile.mkdtemp(prefix="hash_demo_"))
file_path = tmp / "test.txt"
file_path.write_text("중요한 데이터", encoding="utf-8")

def file_hash(path):
    """파일의 SHA-256 해시를 계산합니다."""
    sha256 = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            sha256.update(chunk)
    return sha256.hexdigest()

print(f"\n파일 해시: {file_hash(file_path)[:32]}...")

# 정리
import shutil
shutil.rmtree(tmp, ignore_errors=True)

## 🎯 연습 문제

1. `operator.itemgetter`를 사용해 학생 리스트를 점수순으로 정렬하세요.
2. `reduce`와 `operator.mul`을 사용해 리스트 `[1,2,3,4,5]`의 곱을 계산하세요.
3. `hashlib.sha256()`으로 문자열의 해시값을 계산하고, 원본과 비교해 무결성을 확인하세요.
4. 파일의 SHA-256 해시를 계산하는 함수를 작성하고, 파일 수정 전후의 해시를 비교하세요.